# Predicción de planes telefónicos con Machine Learning

**Proyecto académico de Sprint 10 · TripleTen.** Este notebook presenta código utilizado en un ejercicio formativo; no corresponde a un sistema desplegado en producción.

## Objetivo
Clasificación supervisada para seleccionar un plan telefónico con árbol de decisión, Random Forest y regresión logística; validación de hiperparámetros y comparación con un modelo base.

## Datos y reproducibilidad
Los archivos originales del ejercicio **no están incluidos** en el repositorio. Para ejecutar el análisis, coloca copias obtenidas legítimamente en una carpeta `data/` junto al notebook. Archivos esperados: users_behavior.csv.

Los resultados documentados se extrajeron de la ejecución revisada del notebook original; las salidas de celdas se omitieron de esta edición para preservar un documento de código más ligero. El resultado puede variar si se modifican datos, dependencias o versiones de librerías.


In [ ]:
# Carga de librerias

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
# Carga de Dataframe
path = 'data/users_behavior.csv'

df = pd.read_csv(path)

In [ ]:
# ==========================================
# INSPECCIÓN INICIAL DEL DATAFRAME
# ==========================================

print("=" * 60)
print("Primeras filas")
print("=" * 60)
display(df.head())

print("\n" + "=" * 60)
print("Información general")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("Estadísticas descriptivas")
print("=" * 60)
display(df.describe(include='all'))

print("\n" + "=" * 60)
print("Valores faltantes")
print("=" * 60)

missing = (
    df.isna()
      .sum()
      .to_frame("Missing")
      .assign(Percent=lambda x: (x["Missing"] / len(df) * 100).round(2))
      .query("Missing > 0")
      .sort_values("Percent", ascending=False)
)

if missing.empty:
    print("No hay valores faltantes.")
else:
    display(missing)

print("\n" + "=" * 60)
print("Filas duplicadas")
print("=" * 60)
print(f"Duplicated rows: {df.duplicated().sum()}")

In [ ]:
# revisando registros con cero en variable

zero = df[
    (df['calls'] == 0) &
    (df['messages'] == 0) &
    (df['minutes'] == 0) &
    (df['mb_used'] == 0)
]

cnt = len(zero)
print(f"Usuarios con todas las variables en cero: {cnt}")
display(zero.head())

In [ ]:
# obteniendo la distribucion de variable is_ultra

print(df['is_ultra'].value_counts())
print(df['is_ultra'].value_counts(normalize=True))

In [ ]:
# Grafica de distribucion de la variable is_ultra
df['is_ultra'].value_counts().plot(kind='bar')
plt.title('Distribución de la variable objetivo')
plt.xlabel('Plan')
plt.ylabel('Número de usuarios')
plt.xticks([0,1], ['Smart (0)', 'Ultra (1)'], rotation=0)
plt.show()

In [ ]:
#determinacion de variable features y target

features = df.drop('is_ultra', axis=1)
target = df['is_ultra']

print(features.shape)
print(target.shape)

In [ ]:
# dividiendo los datos
features_train, features_temp, target_train, target_temp = train_test_split(
    features, target, test_size=0.4, random_state=12345
)

features_valid, features_test, target_valid, target_test = train_test_split(
    features_temp, target_temp, test_size=0.5, random_state=12345
)

print(f'train: {features_train.shape}, {target_train.shape}')
print(f'validation: {features_valid.shape}, {target_valid.shape}')
print(f'test: {features_test.shape}, {target_test.shape}')

In [ ]:
#Arbol de Decision

best_model_tree = None
best_score = 0
best_depth = 0

for depth in range(1, 21):

    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=12345
    )

    model.fit(features_train, target_train)

    predictions = model.predict(features_valid)

    score = accuracy_score(target_valid, predictions)

    print(f"Profundidad: {depth:2d} | Accuracy: {score:.3f}")

    if score > best_score:
        best_score = score
        best_depth = depth
        best_model_tree = model

print(f"\nMejor profundidad: {best_depth}")
print(f"Mejor Accuracy: {best_score:.3f}")

In [ ]:
# Bosque aleatorio

best_model_forest = None
best_score = 0
best_est = 0

for est in range(1, 21):

    model = RandomForestClassifier(
        n_estimators=est,
        random_state=12345
    )

    model.fit(features_train, target_train)

    score = model.score(features_valid, target_valid)

    print(f"Árboles: {est:3d} | Accuracy: {score:.3f}")

    if score > best_score:
        best_score = score
        best_est = est
        best_model_forest = model

print(f"\nMejor número de árboles: {best_est}")
print(f"Mejor Accuracy: {best_score:.3f}")

In [ ]:
# Regresion logistica

model = LogisticRegression(
    random_state=12345,
    solver='liblinear'
)

model.fit(features_train, target_train)

score_train = model.score(features_train, target_train)
score_valid = model.score(features_valid, target_valid)

best_model_regresion = model
best_score = score_valid

print(f"Accuracy entrenamiento: {score_train:.3f}")
print(f"Accuracy validación: {score_valid:.3f}")

In [ ]:
test_predictions = best_model_forest.predict(features_test)

# Accuracy final
test_accuracy = accuracy_score(target_test, test_predictions)

print(f"Accuracy en prueba: {test_accuracy:.3f}")

In [ ]:
baseline_accuracy = target_test.value_counts(normalize=True).max()

print(f"Accuracy modelo base: {baseline_accuracy:.3f}")

## Resultado documentado

Random Forest: accuracy de prueba de 0.785, frente a 0.684 de referencia; división de datos 60%/20%/20% en entrenamiento, validación y prueba.

## Alcance
Proyecto académico de Data Science desarrollado durante la formación en TripleTen. El notebook presenta el proceso de análisis, el código y las conclusiones del ejercicio.
